# Convolutional networks on MNIST

An MLP treats an image as a list of 784 unrelated numbers. Shuffle the pixels the same way for every image and it learns just as well, which tells you it ignores the layout. A convolutional layer uses the layout. It slides one small filter over the image, so the same few weights look for the same pattern everywhere. This notebook builds convolution and pooling in NumPy, then compares a Keras CNN with the MLP from notebook 01.

Order: intuition, maths, cost function, from-scratch version, library version, tuning, what breaks.

In [ ]:
from common.dl import get_device, set_seed, setup_keras

setup_keras("torch")
import keras
import matplotlib.pyplot as plt
import numpy as np
import torch

from common.datasets import load_mnist
from common.plotting import plot_history

SEED = 42
set_seed(SEED)
keras.utils.set_random_seed(SEED)
np.set_printoptions(precision=2, suppress=True)
plt.rcParams["figure.figsize"] = (6, 3.5)
print("device:", get_device())
torch.backends.cudnn.deterministic = True  # same seed, same numbers on a GPU

## 1. Intuition

A vertical edge looks the same wherever it sits. One filter that detects it, applied at every position, needs 9 weights instead of one weight per pixel per detector. Two ideas do the work:

- **Local connections.** Each output looks at a small patch, not the whole image.
- **Shared weights.** The same filter is used at every position, so a pattern found in one corner is found in the others.

Pooling then keeps the strongest response in each small block, which halves the size of the map and ignores small shifts inside the block.

## 2. Maths and cost function

For a 2D image $I$ and a $k\times k$ filter $K$, the output at position $(i,j)$ is

$$O_{ij}=\sum_{a=0}^{k-1}\sum_{b=0}^{k-1}K_{ab}\,I_{i+a,\,j+b}+b_0$$

(deep learning calls this convolution, but it is cross-correlation, with no filter flip). A layer with $C_{in}$ input and $C_{out}$ output channels has $(k^2C_{in}+1)\,C_{out}$ weights, whatever the image size. The loss is the same cross-entropy as before. Only the layers change.

## 3. From scratch

A filter that responds to dark-to-bright vertical edges, run over an 8x8 image with an edge down the middle, then max-pooled in 2x2 blocks.

In [ ]:
def conv2d(image: np.ndarray, kernel: np.ndarray) -> np.ndarray:
    """Valid cross-correlation of one 2D image with one 2D kernel (what deep learning calls conv)."""
    kh, kw = kernel.shape
    h, w = image.shape
    out = np.zeros((h - kh + 1, w - kw + 1))
    for i in range(out.shape[0]):
        for j in range(out.shape[1]):
            out[i, j] = (image[i : i + kh, j : j + kw] * kernel).sum()
    return out


def max_pool(x: np.ndarray, size: int = 2) -> np.ndarray:
    h, w = x.shape[0] // size, x.shape[1] // size
    return x[: h * size, : w * size].reshape(h, size, w, size).max(axis=(1, 3))


image = np.zeros((8, 8))
image[:, 4:] = 1.0  # a vertical edge: dark left, bright right
vertical = np.array([[-1, 0, 1], [-1, 0, 1], [-1, 0, 1]])
fmap = conv2d(image, vertical)
print("feature map (6x6), columns:\n", fmap[0])
print("pooled (3x3):\n", max_pool(fmap))
print("same kernel on the edge shifted one pixel right:")
shifted = np.roll(image, 1, axis=1)
shifted[:, 0] = 0
print(
    " feature map row:",
    conv2d(shifted, vertical)[0],
    "pooled row:",
    max_pool(conv2d(shifted, vertical))[0],
)
fig, ax = plt.subplots(1, 3, figsize=(9, 3))
for a, im, t in zip(ax, (image, fmap, max_pool(fmap)), ("image 8x8", "conv 6x6", "pool 3x3")):
    a.imshow(im, cmap="gray")
    a.set_title(t)
    a.axis("off")
plt.show()

The feature map is zero everywhere except two columns at the edge, where it is 3. Pooling keeps the 3 and shrinks the map from 6x6 to 3x3. Shift the edge one pixel and the feature map moves with it, so the pooled map changes as well. Pooling absorbs shifts smaller than its block, not larger ones. Keep this in mind for section 6.

## 4. Library version

Two conv-and-pool blocks (16 then 32 filters, 3x3), then one dense layer with a softmax. We count the parameters by hand and check Keras agrees.

In [ ]:
x_tr, y_tr, x_te, y_te = load_mnist(5000, 1000, seed=SEED)
x_tr4, x_te4 = x_tr[..., None], x_te[..., None]  # (n, 28, 28, 1)


def make_cnn(filters: tuple[int, int] = (16, 32)) -> keras.Model:
    set_seed(SEED)
    keras.utils.set_random_seed(SEED)
    model = keras.Sequential(
        [
            keras.Input((28, 28, 1)),
            keras.layers.Conv2D(filters[0], 3, activation="relu"),
            keras.layers.MaxPooling2D(2),
            keras.layers.Conv2D(filters[1], 3, activation="relu"),
            keras.layers.MaxPooling2D(2),
            keras.layers.Flatten(),
            keras.layers.Dense(10, activation="softmax"),
        ]
    )
    model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model


def make_mlp() -> keras.Model:
    set_seed(SEED)
    keras.utils.set_random_seed(SEED)
    model = keras.Sequential(
        [
            keras.Input((28, 28, 1)),
            keras.layers.Flatten(),
            keras.layers.Dense(128, activation="relu"),
            keras.layers.Dense(10, activation="softmax"),
        ]
    )
    model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model


cnn = make_cnn()
cnn.summary()
by_hand = (3 * 3 * 1 + 1) * 16 + (3 * 3 * 16 + 1) * 32 + (5 * 5 * 32 + 1) * 10
print("parameters by hand:", by_hand, "Keras:", cnn.count_params())

The hand count and Keras agree on 12,810: 160 in the first conv, 4,640 in the second and 8,010 in the dense layer. The MLP from notebook 01 has 101,770, about 8 times more. Now train both for 5 epochs on the same 5,000 images.

In [ ]:
mlp = make_mlp()
h_mlp = mlp.fit(x_tr4, y_tr, epochs=5, batch_size=128, validation_split=0.2, verbose=0)
h_cnn = cnn.fit(x_tr4, y_tr, epochs=5, batch_size=128, validation_split=0.2, verbose=0)
for name, m in (("MLP", mlp), ("CNN", cnn)):
    print(
        f"{name}: {m.count_params():>7} parameters, test accuracy {m.evaluate(x_te4, y_te, verbose=0)[1]:.3f}"
    )
plot_history(h_cnn.history, ("loss", "val_loss"), title="CNN loss")
plt.show()

With one eighth of the parameters the CNN scores 91.7% against 90.0% for the MLP. The gain is small after five epochs on 4,000 training images. It would grow with more data and more training, but it shows the point: structure in the architecture can replace parameters.

## 5. Tuning

Filter counts set the width. Four sizes, same 5 epochs, trained on all 5,000 images.

In [ ]:
for filters in [(4, 8), (8, 16), (16, 32), (32, 64)]:
    m = make_cnn(filters)
    m.fit(x_tr4, y_tr, epochs=5, batch_size=128, verbose=0)
    print(
        f"filters {filters}: {m.count_params():>6} parameters, test accuracy {m.evaluate(x_te4, y_te, verbose=0)[1]:.3f}"
    )

Accuracy climbs from 87.9% with 2,346 parameters to 94.1% with 34,826. Even the smallest CNN is within a few points of the 101,770-parameter MLP. Single runs are noisy at this size (the 16/32 model scores 93.0% here and 91.7% above, with the only change being 1,000 more training images), so read differences under a point or two as noise.

## 6. What breaks

The CNN has weight sharing, so it should cope with a digit that sits somewhere else. Shift the test digits sideways by 0, 2, 4 and 6 pixels.

In [ ]:
def shift(images: np.ndarray, dx: int) -> np.ndarray:
    return np.roll(images, dx, axis=2)  # digits sit in the centre, so the wrap-around is blank


for dx in (0, 2, 4, 6):
    xs = shift(x_te4, dx)
    print(
        f"shift {dx} px: MLP {mlp.evaluate(xs, y_te, verbose=0)[1]:.3f}, CNN {cnn.evaluate(xs, y_te, verbose=0)[1]:.3f}"
    )

Both fall apart. At 2 pixels the MLP drops from 90% to 63% and the CNN from 92% to 78%. At 4 pixels the CNN is at 39% and at 6 pixels the MLP is at 9% and the CNN at 16%, both near chance (10%). The CNN does better, but it is not shift-proof. The final dense layer reads a fixed grid position, so a shifted digit lands on the wrong weights. The convolutions are shift-equivariant, but the classifier on top is not.

The usual fix is to show the network the variation. Train on digits shifted by a random -4 to 4 pixels.

In [ ]:
rng = np.random.default_rng(SEED)
x_aug = np.stack(
    [np.roll(im, int(d), axis=1) for im, d in zip(x_tr4, rng.integers(-4, 5, len(x_tr4)))]
)
cnn_aug = make_cnn()
cnn_aug.fit(x_aug, y_tr, epochs=5, batch_size=128, verbose=0)
for dx in (0, 4):
    print(
        f"shift {dx} px: CNN trained on shifted digits {cnn_aug.evaluate(shift(x_te4, dx), y_te, verbose=0)[1]:.3f}"
    )

A CNN uses a fraction of the weights by sharing them, and still needs examples of every nuisance it is meant to ignore. Next: reuse a CNN someone else trained on a far larger set.